# Zajęcia 8 — lokalna demonstracja API

Usługa demonstracyjna FastAPI ładuje istniejący model z preprocessingiem. Ten notebook nie trenuje, nie rejestruje i nie zmienia aliasów. Recall modelu UCI przy progu 0,5 wynosi 7/794; nie jest to model zatwierdzony do decyzji biznesowych.

## Przygotowanie

Po świeżym klonie wykonaj osobno `uv sync --locked`, następnie `uv run --locked python -m scripts.prepare_lesson8`. Skrypt tworzy nowy izolowany klon, środowisko i artefakty z publicznych danych. Ustaw `BANK_MARKETING_SERVING_EVIDENCE` na zwrócony plik evidence.json. Nie używamy domyślnej bazy głównego projektu.

In [ ]:
import json
import os
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

In [ ]:
from scripts.demo_lesson8 import demonstrate
from scripts.verify_lesson7 import fresh_id

selected = os.environ.get("BANK_MARKETING_SERVING_EVIDENCE")
if not selected:
    raise ValueError("Ustaw BANK_MARKETING_SERVING_EVIDENCE na dowód izolowanego rejestru")
evidence = Path(selected).resolve()
output = root / "artifacts/lesson8" / ("notebook-" + fresh_id())

## HTTP, poprawne i błędne żądanie

Wspólny moduł uruchamia osobny proces na 127.0.0.1, wysyła syntetyczne dane oraz błędne żądanie z duration. Zapisuje surowe logi i zatrzymuje proces także przy błędzie. Wersjonowany kontrakt `/v1/predict` zwraca prawdopodobieństwo yes, decyzję, próg, wersję modelu i request_id. `/health/live` sprawdza proces, `/health/ready` model. Dokumentacja: `/docs` i `/openapi.json`.

In [ ]:
result = demonstrate(evidence, output)
print(
    json.dumps(
        {key: result[key] for key in ("synthetic_request", "response", "invalid_response")},
        indent=2,
        ensure_ascii=False,
    )
)

## Opóźnienie i kontrola niezmienności

30 rozgrzanych żądań dla 1 i 50 rekordów daje lokalny pomiar p50/p95/p99. To demonstracja, nie SLO ani test obciążeniowy. Model jest ładowany raz przy starcie procesu. Pipeline inferencji jest tym samym zapisanym pipeline treningu.

In [ ]:
assert result["status"] == "PASS"
assert result["registry_unchanged"]
assert result["server_stopped"]
print(json.dumps(result["latency"], indent=2))
print("PASS: proces zatrzymany; rejestr i aliasy niezmienione")

## Ograniczenia i niedostępność

Błędy wejścia: 422, niegotowy model: 503, błąd inferencji: 500 bez ścieżek i tracebacków. Logi zawierają tylko metadane. Przy godzinnej niedostępności klient wstrzymuje predykcje i oznacza je jako niedostępne; nie zastępuje błędu decyzją no. Retry stosuje ograniczony exponential backoff; powtórna predykcja na przypiętej wersji jest deterministyczna, ale request_id jest nowy. Nie implementujemy kolejki ani trwałości zleceń. Challenger oznacza wyłącznie demonstrację. Zmiany łamiące kontrakt wymagają nowej ścieżki v2. CI, kontenery, wdrożenie i monitoring pozostają na późniejsze zajęcia.